In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import DBSCAN
from scipy.stats import entropy
import lightgbm as lgb
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import os

# ─────────────────────────────────────────
# Global Plot Config
# ─────────────────────────────────────────
plt.rcParams["figure.figsize"] = (11, 7)
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 18
plt.rcParams['font.weight'] = 'bold'
DPI = 800
PALETTE = ['#1B4F72', '#2E86C1', '#85C1E9', '#F39C12', '#E74C3C', '#27AE60', '#8E44AD', '#16A085']
OUT = 'user-data/outputs'
os.makedirs(OUT, exist_ok=True)

# ─────────────────────────────────────────
# STEP 1 – LOAD DATA
# ─────────────────────────────────────────
print("=" * 60)
print("STEP 1: Loading Dataset")
users         = pd.read_csv('data/users.csv')
restaurants   = pd.read_csv('data/restaurants.csv')
interactions  = pd.read_csv('data/interactions.csv')
context       = pd.read_csv('data/context.csv')
social        = pd.read_csv('data/social_graph.csv')
print(f"  Users: {users.shape} | Restaurants: {restaurants.shape}")
print(f"  Interactions: {interactions.shape} | Context: {context.shape}")

# ─────────────────────────────────────────
# STEP 2 – DATA PREPROCESSING
# ─────────────────────────────────────────
print("\nSTEP 2: Preprocessing")

# Missing values
users.dropna(inplace=True)
restaurants.dropna(inplace=True)
interactions.dropna(inplace=True)

# Merge interaction scores: weight by type
score_map = {'click': 1, 'like': 3, 'visit': 4, 'review': 5}
interactions['score'] = interactions['interaction_type'].map(score_map)

# User-item matrix
ui = interactions.groupby(['user_id', 'restaurant_id'])['score'].sum().reset_index()
ui_matrix = ui.pivot(index='user_id', columns='restaurant_id', values='score').fillna(0)

# Encode categoricals in restaurants
le_cuisine  = LabelEncoder()
le_location = LabelEncoder()
le_price    = LabelEncoder()
restaurants['cuisine_enc']  = le_cuisine.fit_transform(restaurants['cuisine'])
restaurants['location_enc'] = le_location.fit_transform(restaurants['location'])
restaurants['price_enc']    = le_price.fit_transform(restaurants['price_range'])

# Normalize restaurant features
scaler = MinMaxScaler()
feat_cols = ['rating', 'cuisine_enc', 'location_enc', 'price_enc']
restaurants[['rating_norm', 'cuisine_norm', 'location_norm', 'price_norm']] = \
    scaler.fit_transform(restaurants[feat_cols])

print(f"  UI matrix: {ui_matrix.shape} | missing filled with 0")
print(f"  Interaction weights: {score_map}")

# ─────────────────────────────────────────
# STEP 3 – FEATURE ENGINEERING
# ─────────────────────────────────────────
print("\nSTEP 3: Feature Engineering")

# Restaurant popularity score (interaction count normalized)
pop = interactions.groupby('restaurant_id')['score'].agg(['sum','count']).reset_index()
pop.columns = ['restaurant_id','score_sum','interaction_count']
pop['popularity'] = MinMaxScaler().fit_transform(pop[['score_sum']])
restaurants = restaurants.merge(pop[['restaurant_id','popularity','interaction_count']], on='restaurant_id', how='left')
restaurants['popularity'] = restaurants['popularity'].fillna(0)
restaurants['interaction_count'] = restaurants['interaction_count'].fillna(0)

# Cuisine similarity matrix (restaurant × restaurant)
rest_feat = restaurants[['rating_norm','cuisine_norm','location_norm','price_norm']].values
cuisine_sim = cosine_similarity(rest_feat)

# Location proximity feature (encode lat-lon proxy by location group)
loc_coords = {loc: (i*5.0, i*3.0) for i, loc in enumerate(restaurants['location'].unique())}
restaurants['loc_x'] = restaurants['location'].map(lambda x: loc_coords[x][0])
restaurants['loc_y'] = restaurants['location'].map(lambda x: loc_coords[x][1])

print(f"  Popularity scores computed | Cuisine sim matrix: {cuisine_sim.shape}")

# ─────────────────────────────────────────
# STEP 4 – BIAS IDENTIFICATION
# ─────────────────────────────────────────
print("\nSTEP 4: Bias Identification")

def gini_index(values):
    arr = np.sort(np.array(values, dtype=float))
    n = len(arr)
    if arr.sum() == 0:
        return 0.0
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * arr) / (n * arr.sum())) - (n + 1) / n

# Popularity bias (Gini on interaction counts)
pop_counts = restaurants['interaction_count'].fillna(0).values
gini_pop = gini_index(pop_counts)

# Geographic bias (DBSCAN clustering)
coords = restaurants[['loc_x','loc_y']].values
db = DBSCAN(eps=8, min_samples=3).fit(coords)
geo_labels = db.labels_
n_clusters = len(set(geo_labels)) - (1 if -1 in geo_labels else 0)
geo_bias_ratio = np.sum(geo_labels == geo_labels[geo_labels != -1].max()) / len(geo_labels) if n_clusters > 0 else 0

# Cuisine bias (Shannon Entropy)
cuisine_dist = restaurants['cuisine'].value_counts(normalize=True).values
cuisine_entropy = entropy(cuisine_dist, base=2)
max_entropy = np.log2(len(cuisine_dist))
cuisine_bias = 1 - (cuisine_entropy / max_entropy)

# Exposure bias (distribution imbalance: top 20% restaurants getting what % of interactions)
sorted_counts = np.sort(pop_counts)[::-1]
top20_idx = int(len(sorted_counts) * 0.2)
exposure_bias = sorted_counts[:top20_idx].sum() / (sorted_counts.sum() + 1e-9)

bias_metrics = {
    'Popularity Bias (Gini)': round(gini_pop, 4),
    'Geographic Bias (Cluster Ratio)': round(geo_bias_ratio, 4),
    'Cuisine Bias (1 - Entropy Ratio)': round(cuisine_bias, 4),
    'Exposure Bias (Top-20% Share)': round(exposure_bias, 4),
}
print(f"  {bias_metrics}")

# ─────────────────────────────────────────
# STEP 5 – INITIAL RECOMMENDATION (Hybrid)
# ─────────────────────────────────────────
print("\nSTEP 5: Hybrid Recommendation Model")

# --- Collaborative Filtering via SVD ---
svd = TruncatedSVD(n_components=20, random_state=42)
U = svd.fit_transform(ui_matrix.values)
Vt = svd.components_
cf_scores_matrix = pd.DataFrame(
    np.dot(U, Vt),
    index=ui_matrix.index,
    columns=ui_matrix.columns
)

# --- Content-Based Filtering ---
# User preference vector: average of all restaurants they interacted with, weighted by score
user_pref = {}
for uid, grp in interactions.groupby('user_id'):
    rids = grp['restaurant_id'].values
    scores = grp['score'].values
    rfeats = restaurants.set_index('restaurant_id')[['rating_norm','cuisine_norm','location_norm','price_norm']]
    valid = [r for r in rids if r in rfeats.index]
    if not valid:
        user_pref[uid] = np.zeros(4)
        continue
    wscores = np.array([scores[i] for i, r in enumerate(rids) if r in rfeats.index])
    feats   = rfeats.loc[valid].values
    user_pref[uid] = np.average(feats, axis=0, weights=wscores / wscores.sum())

rest_feat_df = restaurants.set_index('restaurant_id')[['rating_norm','cuisine_norm','location_norm','price_norm']]

def cb_score_for_user(uid):
    if uid not in user_pref:
        return pd.Series(0, index=rest_feat_df.index)
    pref = user_pref[uid].reshape(1, -1)
    sims = cosine_similarity(pref, rest_feat_df.values)[0]
    return pd.Series(sims, index=rest_feat_df.index)

# --- LightGBM Hybrid Ranking ---
# Build training set: label = interaction score
all_users = ui_matrix.index.tolist()
sample_users = np.random.RandomState(42).choice(all_users, size=min(300, len(all_users)), replace=False)

rows = []
for uid in sample_users:
    cf_row  = cf_scores_matrix.loc[uid] if uid in cf_scores_matrix.index else pd.Series(dtype=float)
    cb_row  = cb_score_for_user(uid)
    user_ints = interactions[interactions['user_id'] == uid]
    for _, r in user_ints.iterrows():
        rid = r['restaurant_id']
        if rid not in rest_feat_df.index:
            continue
        cf_s = cf_row[rid] if rid in cf_row.index else 0
        cb_s = cb_row[rid] if rid in cb_row.index else 0
        pop_s = restaurants.set_index('restaurant_id')['popularity'].get(rid, 0)
        rat_s = restaurants.set_index('restaurant_id')['rating_norm'].get(rid, 0)
        rows.append({'user_id': uid, 'restaurant_id': rid,
                     'cf_score': cf_s, 'cb_score': cb_s,
                     'popularity': pop_s, 'rating_norm': rat_s,
                     'label': r['score']})

train_df = pd.DataFrame(rows).dropna()
X = train_df[['cf_score','cb_score','popularity','rating_norm']].values
y = train_df['label'].values

lgb_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42, verbose=-1)
lgb_model.fit(X, y)
print(f"  LightGBM trained on {len(train_df)} samples")

def get_initial_recs(uid, top_k=10):
    rest_ids = rest_feat_df.index.tolist()
    cf_row = cf_scores_matrix.loc[uid] if uid in cf_scores_matrix.index else pd.Series(0, index=rest_ids)
    cb_row = cb_score_for_user(uid)
    pop_s  = restaurants.set_index('restaurant_id')['popularity']
    rat_s  = restaurants.set_index('restaurant_id')['rating_norm']
    Xpred = np.array([[
        cf_row.get(r, 0),
        cb_row.get(r, 0),
        pop_s.get(r, 0),
        rat_s.get(r, 0)
    ] for r in rest_ids])
    preds = lgb_model.predict(Xpred)
    rec_df = pd.DataFrame({'restaurant_id': rest_ids, 'relevance_score': preds})
    rec_df = rec_df.sort_values('relevance_score', ascending=False).head(top_k * 5)
    return rec_df

# ─────────────────────────────────────────
# STEP 6 – BIAS MITIGATION: MMR RE-RANKING
# ─────────────────────────────────────────
print("\nSTEP 6: MMR Diversity-Aware Re-ranking")

LAMBDA = 0.6  # trade-off: relevance vs diversity

def diversity_score(rid, selected_ids, sim_matrix, rid_to_idx):
    if not selected_ids:
        return 1.0
    idx_r = rid_to_idx.get(rid)
    if idx_r is None:
        return 1.0
    sims = [sim_matrix[idx_r, rid_to_idx[s]] for s in selected_ids if s in rid_to_idx]
    return 1 - (max(sims) if sims else 0)

# Build restaurant index map for sim matrix
rest_ids_all = restaurants['restaurant_id'].tolist()
rid_to_idx = {r: i for i, r in enumerate(rest_ids_all)}

def mmr_rerank(rec_df, top_k=10):
    candidates = rec_df.copy().reset_index(drop=True)
    s_min = candidates['relevance_score'].min()
    s_max = candidates['relevance_score'].max()
    candidates['relevance_norm'] = (candidates['relevance_score'] - s_min) / (s_max - s_min + 1e-9)
    selected = []
    remaining = candidates['restaurant_id'].tolist()
    while len(selected) < top_k and remaining:
        best, best_score = None, -np.inf
        for rid in remaining:
            rel = candidates.loc[candidates['restaurant_id'] == rid, 'relevance_norm'].values[0]
            div = diversity_score(rid, selected, cuisine_sim, rid_to_idx)
            mmr = LAMBDA * rel + (1 - LAMBDA) * div
            if mmr > best_score:
                best, best_score = rid, mmr
        selected.append(best)
        remaining.remove(best)
    return selected

# ─────────────────────────────────────────
# STEP 7 – EVALUATION
# ─────────────────────────────────────────
print("\nSTEP 7: Evaluation")

K = 10
eval_users = np.random.RandomState(0).choice(all_users, size=min(100, len(all_users)), replace=False)

def precision_at_k(recommended, relevant, k):
    rec_k = recommended[:k]
    return len(set(rec_k) & set(relevant)) / k

def recall_at_k(recommended, relevant, k):
    if not relevant:
        return 0.0
    return len(set(recommended[:k]) & set(relevant)) / len(relevant)

def ndcg_at_k(recommended, relevant_scores, k):
    dcg, idcg = 0.0, 0.0
    for i, r in enumerate(recommended[:k]):
        rel = relevant_scores.get(r, 0)
        dcg += rel / np.log2(i + 2)
    ideal = sorted(relevant_scores.values(), reverse=True)[:k]
    for i, rel in enumerate(ideal):
        idcg += rel / np.log2(i + 2)
    return dcg / idcg if idcg > 0 else 0

def diversity_metric(recs, sim_matrix, rid_to_idx):
    if len(recs) < 2:
        return 0.0
    sims = []
    for i in range(len(recs)):
        for j in range(i+1, len(recs)):
            ia, ib = rid_to_idx.get(recs[i]), rid_to_idx.get(recs[j])
            if ia is not None and ib is not None:
                sims.append(sim_matrix[ia, ib])
    return 1 - np.mean(sims) if sims else 0.0

results_before, results_after = [], []

for uid in eval_users:
    rec_df = get_initial_recs(uid, top_k=K)
    initial_recs = rec_df['restaurant_id'].tolist()[:K]
    mmr_recs = mmr_rerank(rec_df, top_k=K)

    user_ints = interactions[interactions['user_id'] == uid]
    relevant = user_ints[user_ints['score'] >= 3]['restaurant_id'].tolist()
    rel_scores = dict(zip(user_ints['restaurant_id'], user_ints['score']))

    # Gini of recommended item popularity
    def local_gini(recs):
        cnts = restaurants.set_index('restaurant_id')['interaction_count']
        vals = [cnts.get(r, 0) for r in recs]
        return gini_index(vals)

    results_before.append({
        'precision': precision_at_k(initial_recs, relevant, K),
        'recall': recall_at_k(initial_recs, relevant, K),
        'ndcg': ndcg_at_k(initial_recs, rel_scores, K),
        'diversity': diversity_metric(initial_recs, cuisine_sim, rid_to_idx),
        'gini': local_gini(initial_recs),
    })
    results_after.append({
        'precision': precision_at_k(mmr_recs, relevant, K),
        'recall': recall_at_k(mmr_recs, relevant, K),
        'ndcg': ndcg_at_k(mmr_recs, rel_scores, K),
        'diversity': diversity_metric(mmr_recs, cuisine_sim, rid_to_idx),
        'gini': local_gini(mmr_recs),
    })

bf = pd.DataFrame(results_before).mean()
af = pd.DataFrame(results_after).mean()

# Coverage
all_recs_before = set()
all_recs_after  = set()
for uid in eval_users:
    rec_df = get_initial_recs(uid, top_k=K)
    all_recs_before.update(rec_df['restaurant_id'].tolist()[:K])
    all_recs_after.update(mmr_rerank(rec_df, top_k=K))

cov_before = len(all_recs_before) / len(restaurants)
cov_after  = len(all_recs_after)  / len(restaurants)

print("\n  ── Evaluation Summary ──")
print(f"  {'Metric':<30} {'Before':>10} {'After':>10}")
for m in ['precision','recall','ndcg','diversity','gini']:
    print(f"  {m:<30} {bf[m]:>10.4f} {af[m]:>10.4f}")
print(f"  {'coverage':<30} {cov_before:>10.4f} {cov_after:>10.4f}")

# ─────────────────────────────────────────
# ── PLOTS (7) ──
# ─────────────────────────────────────────
print("\nGenerating Plots...")

# ── Plot 1: Dataset Overview ──
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle('Dataset Overview', fontsize=20, fontweight='bold', y=1.02)

# 1a cuisine dist
cuis_cnt = restaurants['cuisine'].value_counts()
axes[0].bar(cuis_cnt.index, cuis_cnt.values, color=PALETTE[:len(cuis_cnt)])
axes[0].set_title('Restaurant Cuisine Distribution', fontweight='bold')
axes[0].set_xlabel('Cuisine')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# 1b interaction types
int_cnt = interactions['interaction_type'].value_counts()
axes[1].pie(int_cnt.values, labels=int_cnt.index, autopct='%1.1f%%',
            colors=PALETTE[:len(int_cnt)], startangle=140,
            textprops={'fontsize': 16, 'fontweight': 'bold'})
axes[1].set_title('Interaction Type Distribution', fontweight='bold')

# 1c user preferred cuisine
uc = users['preferred_cuisine'].value_counts()
axes[2].barh(uc.index, uc.values, color=PALETTE[:len(uc)])
axes[2].set_title('User Preferred Cuisine', fontweight='bold')
axes[2].set_xlabel('Count')

plt.tight_layout()
plt.savefig(f'{OUT}/plot1_dataset_overview.png', dpi=DPI, bbox_inches='tight')
plt.close()
print("  Plot 1 saved")

# ── Plot 2: Bias Identification ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Bias Identification Analysis', fontsize=20, fontweight='bold')

# 2a Popularity (Lorenz Curve)
sorted_pop = np.sort(pop_counts)
cum_pop = np.cumsum(sorted_pop) / sorted_pop.sum()
cum_rest = np.linspace(0, 1, len(sorted_pop))
axes[0,0].plot(cum_rest, cum_pop, color=PALETTE[0], linewidth=2.5, label=f'Lorenz Curve (Gini={gini_pop:.3f})')
axes[0,0].plot([0,1],[0,1],'--', color='gray', linewidth=1.5, label='Perfect Equality')
axes[0,0].fill_between(cum_rest, cum_pop, cum_rest, alpha=0.2, color=PALETTE[0])
axes[0,0].set_title('Popularity Bias – Lorenz Curve', fontweight='bold')
axes[0,0].set_xlabel('Cumulative Restaurants')
axes[0,0].set_ylabel('Cumulative Interactions')
axes[0,0].legend(fontsize=14)

# 2b Geographic distribution
loc_cnt = restaurants['location'].value_counts()
axes[0,1].bar(loc_cnt.index, loc_cnt.values, color=PALETTE[1])
axes[0,1].set_title('Geographic Bias – Restaurant Distribution', fontweight='bold')
axes[0,1].set_xlabel('City')
axes[0,1].set_ylabel('Count')
axes[0,1].tick_params(axis='x', rotation=30)

# 2c Cuisine entropy
axes[1,0].bar(cuis_cnt.index, cuis_cnt.values / cuis_cnt.sum(), color=PALETTE[2])
axes[1,0].set_title(f'Cuisine Bias – Entropy={cuisine_entropy:.3f} / Max={max_entropy:.3f}', fontweight='bold')
axes[1,0].set_xlabel('Cuisine')
axes[1,0].set_ylabel('Proportion')
axes[1,0].tick_params(axis='x', rotation=45)

# 2d Exposure bias (top-N share)
thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
shares = []
for th in thresholds:
    top_idx = int(len(sorted_pop) * th)
    share = np.sort(pop_counts)[::-1][:top_idx].sum() / (pop_counts.sum() + 1e-9)
    shares.append(share)
axes[1,1].plot([t*100 for t in thresholds], shares, 'o-', color=PALETTE[3], linewidth=2.5, markersize=8)
axes[1,1].axhline(y=0.8, color='red', linestyle='--', linewidth=1.5, label='80% Threshold')
axes[1,1].set_title('Exposure Bias – Top-N% Interaction Share', fontweight='bold')
axes[1,1].set_xlabel('Top N% Restaurants')
axes[1,1].set_ylabel('Interaction Share')
axes[1,1].legend(fontsize=14)

plt.tight_layout()
plt.savefig(f'{OUT}/plot2_bias_identification.png', dpi=DPI, bbox_inches='tight')
plt.close()
print("  Plot 2 saved")

# ── Plot 3: Model Feature Importance ──
fig, ax = plt.subplots(figsize=(11, 7))
feat_names = ['CF Score', 'CB Score', 'Popularity', 'Rating']
importances = lgb_model.feature_importances_
order = np.argsort(importances)
ax.barh([feat_names[i] for i in order], importances[order], color=PALETTE[:4])
ax.set_title('LightGBM Feature Importance', fontweight='bold')
ax.set_xlabel('Importance Score')
for i, (idx, val) in enumerate(zip(order, importances[order])):
    ax.text(val + 0.5, i, f'{val:.0f}', va='center', fontweight='bold', fontsize=15)
plt.tight_layout()
plt.savefig(f'{OUT}/plot3_feature_importance.png', dpi=DPI, bbox_inches='tight')
plt.close()
print("  Plot 3 saved")

# ── Plot 4: Accuracy Metrics Before vs After ──
fig, ax = plt.subplots(figsize=(11, 7))
metrics = ['Precision@10', 'Recall@10', 'NDCG@10']
before_vals = [bf['precision'], bf['recall'], bf['ndcg']]
after_vals  = [af['precision'], af['recall'], af['ndcg']]
x = np.arange(len(metrics))
w = 0.35
b1 = ax.bar(x - w/2, before_vals, w, label='Before Re-ranking', color=PALETTE[0])
b2 = ax.bar(x + w/2, after_vals,  w, label='After Re-ranking',  color=PALETTE[3])
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylabel('Score'); ax.set_title('Accuracy Metrics: Before vs After MMR Re-ranking', fontweight='bold')
ax.legend()
for bar in [*b1, *b2]:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.3f}', ha='center', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUT}/plot4_accuracy_comparison.png', dpi=DPI, bbox_inches='tight')
plt.close()
print("  Plot 4 saved")

# ── Plot 5: Fairness & Diversity Metrics ──
fig, ax = plt.subplots(figsize=(11, 7))
fmetrics = ['Diversity', 'Coverage', 'Gini (lower=better)']
before_f = [bf['diversity'], cov_before, bf['gini']]
after_f  = [af['diversity'], cov_after,  af['gini']]
x = np.arange(len(fmetrics))
b1 = ax.bar(x - w/2, before_f, w, label='Before Re-ranking', color=PALETTE[1])
b2 = ax.bar(x + w/2, after_f,  w, label='After Re-ranking',  color=PALETTE[4])
ax.set_xticks(x); ax.set_xticklabels(fmetrics)
ax.set_ylabel('Score'); ax.set_title('Fairness & Diversity Metrics: Before vs After MMR', fontweight='bold')
ax.legend()
for bar in [*b1, *b2]:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUT}/plot5_fairness_diversity.png', dpi=DPI, bbox_inches='tight')
plt.close()
print("  Plot 5 saved")

# ── Plot 6: Recommendation Score Distribution ──
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Recommendation Score Distribution', fontweight='bold', fontsize=20)

# Before: relevance scores
sample_uid = eval_users[0]
rec_df_sample = get_initial_recs(sample_uid, top_k=K)
axes[0].hist(rec_df_sample['relevance_score'], bins=20, color=PALETTE[0], edgecolor='white')
axes[0].set_title('Before MMR – Relevance Score Distribution', fontweight='bold')
axes[0].set_xlabel('Relevance Score'); axes[0].set_ylabel('Frequency')

# After: show diversity scores across all eval users
all_divs_before = [r['diversity'] for r in results_before]
all_divs_after  = [r['diversity'] for r in results_after]
axes[1].hist(all_divs_before, bins=15, alpha=0.7, label='Before', color=PALETTE[0], edgecolor='white')
axes[1].hist(all_divs_after,  bins=15, alpha=0.7, label='After',  color=PALETTE[3], edgecolor='white')
axes[1].set_title('Diversity Score Distribution Across Users', fontweight='bold')
axes[1].set_xlabel('Diversity Score'); axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{OUT}/plot6_score_distribution.png', dpi=DPI, bbox_inches='tight')
plt.close()
print("  Plot 6 saved")

# ── Plot 7: Cuisine Coverage Heatmap ──
# How much each cuisine is represented in recommendations before vs after
from collections import Counter

def cuisine_coverage(eval_users, use_mmr=False):
    counter = Counter()
    rid_to_cuisine = restaurants.set_index('restaurant_id')['cuisine'].to_dict()
    for uid in eval_users:
        rec_df = get_initial_recs(uid, top_k=K)
        recs = mmr_rerank(rec_df, top_k=K) if use_mmr else rec_df['restaurant_id'].tolist()[:K]
        for r in recs:
            c = rid_to_cuisine.get(r, 'Unknown')
            counter[c] += 1
    return counter

cc_before = cuisine_coverage(eval_users, use_mmr=False)
cc_after  = cuisine_coverage(eval_users, use_mmr=True)
all_cuisines = sorted(set(list(cc_before.keys()) + list(cc_after.keys())))
before_arr = np.array([cc_before.get(c, 0) for c in all_cuisines])
after_arr  = np.array([cc_after.get(c, 0)  for c in all_cuisines])

fig, ax = plt.subplots(figsize=(13, 7))
x = np.arange(len(all_cuisines))
ax.bar(x - w/2, before_arr, w, label='Before Re-ranking', color=PALETTE[0])
ax.bar(x + w/2, after_arr,  w, label='After Re-ranking',  color=PALETTE[3])
ax.set_xticks(x); ax.set_xticklabels(all_cuisines, rotation=30, ha='right')
ax.set_title('Cuisine Coverage in Recommendations: Before vs After MMR', fontweight='bold')
ax.set_xlabel('Cuisine'); ax.set_ylabel('Recommendation Count')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUT}/plot7_cuisine_coverage.png', dpi=DPI, bbox_inches='tight')
plt.close()
print("  Plot 7 saved")

# ─────────────────────────────────────────
# ── EXCEL TABLES (7) ──
# ─────────────────────────────────────────
print("\nGenerating Excel Tables...")

wb = Workbook()

def style_header(cell, bg='1B4F72'):
    cell.font = Font(bold=True, color='FFFFFF', name='Times New Roman', size=13)
    cell.fill = PatternFill('solid', start_color=bg)
    cell.alignment = Alignment(horizontal='center', vertical='center')

def style_row(cell, alt=False):
    cell.font = Font(name='Times New Roman', size=12)
    cell.alignment = Alignment(horizontal='center', vertical='center')
    if alt:
        cell.fill = PatternFill('solid', start_color='D6EAF8')

def style_title(ws, title, merge_to):
    ws.merge_cells(f'A1:{merge_to}1')
    ws['A1'] = title
    ws['A1'].font = Font(bold=True, size=15, name='Times New Roman', color='1B4F72')
    ws['A1'].alignment = Alignment(horizontal='center', vertical='center')
    ws.row_dimensions[1].height = 30

def add_table(ws, headers, rows, start_row=2):
    for col, h in enumerate(headers, 1):
        c = ws.cell(row=start_row, column=col, value=h)
        style_header(c)
        ws.column_dimensions[get_column_letter(col)].width = max(18, len(str(h)) + 4)
    for ri, row in enumerate(rows, start_row + 1):
        alt = ri % 2 == 0
        for ci, val in enumerate(row, 1):
            c = ws.cell(row=ri, column=ci, value=val)
            style_row(c, alt)

# ── Table 1: Dataset Summary ──
ws1 = wb.active; ws1.title = "T1_Dataset_Summary"
style_title(ws1, "Table 1: Dataset Summary Statistics", "D")
t1_headers = ["Dataset", "Records", "Features", "Key Attributes"]
t1_rows = [
    ["Users", len(users), 4, "user_id, age, gender, preferred_cuisine"],
    ["Restaurants", len(restaurants), 6, "restaurant_id, name, cuisine, rating, price_range, location"],
    ["Interactions", len(interactions), 4, "user_id, restaurant_id, interaction_type, timestamp"],
    ["Context", len(context), 4, "interaction_id, location, time_of_day, weather"],
    ["Social Graph", len(social), 2, "user_a, user_b"],
]
add_table(ws1, t1_headers, t1_rows)

# ── Table 2: Interaction Distribution ──
ws2 = wb.create_sheet("T2_Interaction_Stats")
style_title(ws2, "Table 2: Interaction Type Distribution & Weights", "E")
int_stats = interactions.groupby('interaction_type').agg(
    Count=('score', 'count'), MeanScore=('score', 'mean'), TotalScore=('score', 'sum')
).reset_index()
t2_headers = ["Interaction Type", "Count", "Proportion (%)", "Weight", "Total Score"]
t2_rows = []
for _, r in int_stats.iterrows():
    t2_rows.append([
        r['interaction_type'],
        int(r['Count']),
        round(r['Count'] / len(interactions) * 100, 2),
        score_map.get(r['interaction_type'], 0),
        int(r['TotalScore'])
    ])
add_table(ws2, t2_headers, t2_rows)

# ── Table 3: Bias Metrics ──
ws3 = wb.create_sheet("T3_Bias_Metrics")
style_title(ws3, "Table 3: Bias Identification Metrics", "D")
t3_headers = ["Bias Type", "Metric Used", "Value", "Interpretation"]
t3_rows = [
    ["Popularity Bias", "Gini Index", round(gini_pop, 4), "0=Equal, 1=Highly Concentrated"],
    ["Geographic Bias", "DBSCAN Cluster Ratio", round(geo_bias_ratio, 4), "Higher = More Geographic Concentration"],
    ["Cuisine Bias", "1 - Entropy Ratio", round(cuisine_bias, 4), "0=Uniform, 1=Completely Biased"],
    ["Exposure Bias", "Top-20% Interaction Share", round(exposure_bias, 4), "Share of interactions by top 20% popular restaurants"],
    ["Max Cuisine Entropy", "Shannon Entropy (bits)", round(max_entropy, 4), "Maximum achievable entropy"],
    ["Actual Cuisine Entropy", "Shannon Entropy (bits)", round(cuisine_entropy, 4), "Observed cuisine entropy"],
]
add_table(ws3, t3_headers, t3_rows)

# ── Table 4: Model Configuration ──
ws4 = wb.create_sheet("T4_Model_Config")
style_title(ws4, "Table 4: Hybrid Recommendation Model Configuration", "C")
t4_headers = ["Component", "Method / Parameters", "Purpose"]
t4_rows = [
    ["Collaborative Filtering", f"TruncatedSVD (n_components=20)", "Capture latent user-item interaction patterns"],
    ["Content-Based Filtering", "Cosine Similarity on (rating, cuisine, location, price)", "Match restaurant features to user preferences"],
    ["Hybrid Ranker", "LightGBM (n_est=200, lr=0.05, depth=5)", "Combine CF + CB signals for final ranking"],
    ["Training Samples", str(len(train_df)), "User-restaurant interaction pairs used for training"],
    ["MMR Lambda", str(LAMBDA), "Trade-off: relevance vs diversity (0=diversity only, 1=relevance only)"],
    ["Top-K", str(K), "Number of recommendations per user"],
]
add_table(ws4, t4_headers, t4_rows)

# ── Table 5: Evaluation Before vs After ──
ws5 = wb.create_sheet("T5_Evaluation_Results")
style_title(ws5, "Table 5: Evaluation Results – Before vs After MMR Re-ranking", "D")
t5_headers = ["Metric", "Before Re-ranking", "After Re-ranking", "Change (%)"]
def pct_change(b, a):
    if b == 0: return "N/A"
    return round((a - b) / abs(b) * 100, 2)
t5_rows = [
    ["Precision@10", round(bf['precision'],4), round(af['precision'],4), pct_change(bf['precision'], af['precision'])],
    ["Recall@10",    round(bf['recall'],4),    round(af['recall'],4),    pct_change(bf['recall'], af['recall'])],
    ["NDCG@10",      round(bf['ndcg'],4),      round(af['ndcg'],4),      pct_change(bf['ndcg'], af['ndcg'])],
    ["Diversity",    round(bf['diversity'],4), round(af['diversity'],4), pct_change(bf['diversity'], af['diversity'])],
    ["Coverage",     round(cov_before,4),      round(cov_after,4),       pct_change(cov_before, cov_after)],
    ["Gini Index",   round(bf['gini'],4),      round(af['gini'],4),      pct_change(bf['gini'], af['gini'])],
]
add_table(ws5, t5_headers, t5_rows)

# ── Table 6: Cuisine Coverage ──
ws6 = wb.create_sheet("T6_Cuisine_Coverage")
style_title(ws6, "Table 6: Cuisine Coverage in Recommendations", "D")
t6_headers = ["Cuisine", "Before Re-ranking", "After Re-ranking", "Coverage Change"]
t6_rows = [[c, cc_before.get(c,0), cc_after.get(c,0), cc_after.get(c,0) - cc_before.get(c,0)]
            for c in all_cuisines]
add_table(ws6, t6_headers, t6_rows)

# ── Table 7: Restaurant Popularity Distribution ──
ws7 = wb.create_sheet("T7_Popularity_Distribution")
style_title(ws7, "Table 7: Restaurant Popularity Distribution", "E")
t7_headers = ["Decile", "Restaurants", "Avg Interactions", "Avg Popularity", "Share of Total Interactions (%)"]
decile_data = []
pop_df = restaurants[['restaurant_id','interaction_count','popularity']].copy()
pop_df['decile'] = pd.qcut(pop_df['interaction_count'], q=10, labels=False, duplicates='drop')
for d, grp in pop_df.groupby('decile'):
    share = grp['interaction_count'].sum() / pop_df['interaction_count'].sum() * 100
    decile_data.append([
        f"Decile {int(d)+1}",
        len(grp),
        round(grp['interaction_count'].mean(), 2),
        round(grp['popularity'].mean(), 4),
        round(share, 2)
    ])
add_table(ws7, t7_headers, decile_data)

excel_path = f'{OUT}/restaurant_recommendation_tables.xlsx'
wb.save(excel_path)
print(f"  Excel saved: {excel_path}")

print("\n✅ All 7 plots and 7 tables generated successfully!")
print(f"   Output directory: {OUT}")